# 04. Difference Array

Difference array is the **opposite** of prefix sum. Prefix sum makes **range queries** fast. Difference array makes **range updates** fast.

**Learning order:** REAL LIFE → THE PROBLEM → IDEA → CODE → BOOKINGS → CAR POOLING → INTERVIEW THINKING

## 1. Real life: hotel rooms

A hotel has bookings: "2 rooms from day 1 to day 3", "3 rooms from day 2 to day 4". How many rooms are occupied on each day?

Brute force: for every booking, add to every day in its range → O(bookings × days).

Smarter: at **check-in** day add `+rooms`, on the day **after check-out** add `-rooms`. Then take a running total across days. That is the occupancy.

## 2. The idea

To add `x` to every element in `[l, r]`:

```
diff[l]     += x    # from l onwards, everything is +x
diff[r + 1] -= x    # after r, cancel it
```

Only **two** changes per update, O(1). After all updates, do one prefix sum over `diff` to get the final array.

**Why it works:** `diff[i] = a[i] - a[i-1]` (how much the value *changes* at `i`). Prefix sum of `diff` gives back `a`.

In [1]:
def apply_updates(n, updates):
    diff = [0] * (n + 1)              # +1 so diff[r + 1] is always safe
    for l, r, x in updates:
        diff[l] += x
        diff[r + 1] -= x
    result, running = [], 0
    for i in range(n):
        running += diff[i]
        result.append(running)
    return result

# n=5, add 2 to [1..3], add 3 to [2..4], add -2 to [0..2]
print(apply_updates(5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]]))   # [-2, 0, 3, 5, 3]

[-2, 0, 3, 5, 3]


In [2]:
# Build a difference array from an existing array and rebuild the array
a = [3, 5, 4, 8]
diff = [a[0]] + [a[i] - a[i - 1] for i in range(1, len(a))]
print(diff)                                   # [3, 2, -1, 4]
from itertools import accumulate
print(list(accumulate(diff)))                 # [3, 5, 4, 8]  (back to the original)

[3, 2, -1, 4]
[3, 5, 4, 8]


## 3. Corporate Flight Bookings

Each booking `[first, last, seats]` (1-indexed) adds `seats` to every flight in that range. Return seats booked per flight. Direct use of the difference array. Convert to 0-index: `diff[first-1] += seats`, `diff[last] -= seats`.

In [3]:
def corp_flight_bookings(bookings, n):
    diff = [0] * (n + 1)
    for first, last, seats in bookings:
        diff[first - 1] += seats
        diff[last] -= seats
    result, running = [], 0
    for i in range(n):
        running += diff[i]
        result.append(running)
    return result

print(corp_flight_bookings([[1, 2, 10], [2, 3, 20], [2, 5, 25]], 5))   # [10, 55, 45, 25, 25]

[10, 55, 45, 25, 25]


## 4. Car Pooling (are we over capacity at any point?)

Trip `[passengers, from, to]`: passengers sit from `from` up to (not including) `to`. So `+p` at `from`, `-p` at `to`. If the running total ever exceeds `capacity`, return `False`.

Same idea works for "maximum number of overlapping meetings" (rooms needed).

In [4]:
def car_pooling(trips, capacity):
    last = max(t for _, _, t in trips)
    diff = [0] * (last + 1)
    for p, f, t in trips:
        diff[f] += p
        diff[t] -= p
    running = 0
    for x in diff:
        running += x
        if running > capacity:
            return False
    return True

print(car_pooling([[2, 1, 5], [3, 3, 7]], 4))   # False
print(car_pooling([[2, 1, 5], [3, 3, 7]], 5))   # True

False
True


## 5. Interview thinking

**Signals:** "many updates on ranges, then ask for the final array", "bookings", "intervals overlap count", "add x from l to r".

**Say:** "Brute force touches every element of every range, O(q × n). With a difference array each update is O(1), and one prefix sum at the end gives the result: O(n + q)."

**Limit:** if you must read values **between** updates, difference array is not enough. Use a Fenwick tree or segment tree.

**Common mistakes:** array size `n` instead of `n + 1` (crash at `r + 1`), forgetting the final prefix sum, 1-indexed vs 0-indexed bookings, wrong end boundary (`to` is exclusive in Car Pooling).

## Quick Revision

- Update `[l, r] += x`: `diff[l] += x`, `diff[r+1] -= x`.
- Final array = prefix sum of `diff`.
- Size `n + 1`. Check if the end is inclusive or exclusive.
- Prefix sum = fast queries. Difference array = fast updates. They are inverses.